In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score,recall_score, f1_score
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
df = pd.read_csv("employee_turnover.csv")

In [3]:
#Explore data
df.head()
df.info()
df.shape
df.describe()
df.sample()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1350 entries, 0 to 1349
Data columns (total 16 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Job_Satisfaction                         1350 non-null   float64
 1   Performance_Rating                       1350 non-null   float64
 2   Years_At_Company                         1350 non-null   float64
 3   Work_Life_Balance                        1350 non-null   float64
 4   Distance_From_Home                       1350 non-null   float64
 5   Monthly_Income                           1350 non-null   float64
 6   Education_Level                          1350 non-null   float64
 7   Age                                      1350 non-null   float64
 8   Num_Companies_Worked                     1350 non-null   float64
 9   Employee_Role                            1350 non-null   float64
 10  Annual_Bonus                             1350 no

,Job_Satisfaction,Performance_Rating,Years_At_Company,Work_Life_Balance,Distance_From_Home,Monthly_Income,Education_Level,Age,Num_Companies_Worked,Employee_Role,Annual_Bonus,Training_Hours,Department,Annual_Bonus_Squared,Annual_Bonus_Training_Hours_Interaction,Employee_Turnover
165,0.499909,0.578799,0.185521,0.95084,0.974031,0.190875,0.48137,0.276864,0.529946,0.461096,0.476223,0.871998,0.981178,0.226788,0.415265,1


In [19]:
#Test train split
X = df.drop(["Employee_Turnover"], axis = 1)
y = df["Employee_Turnover"]

X_train_unscale, X_test_unscale, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [21]:
# Check target feature value distribution
y_train[y_train == 1] # 547
y_train[y_train == 0] # 533

755     0
109     0
1040    0
774     0
983     0
       ..
330     0
1238    0
466     0
121     0
1044    0
Name: Employee_Turnover, Length: 533, dtype: int64

In [22]:
# Train model: Logistic Regression
logistic_regr_model = LogisticRegression()
logistic_regr_model.fit(X_train_unscale, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [23]:
# predict
y_pred = logistic_regr_model.predict(X_test_unscale)

In [24]:
# Evaluate model by comparing predicted vs actual values
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"Accuracy:{accuracy} \n Precision:{precision} \n Recall:{recall} \n F1_score:{f1}")

Accuracy:0.8592592592592593 
 Precision:0.8717948717948718 
 Recall:0.816 
 F1_score:0.8429752066115702


In [25]:
#Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_unscale)
X_test = scaler.transform(X_test_unscale)

In [26]:
scaled_logit_model = LogisticRegression()
scaled_logit_model.fit(X_train, y_train)
y_pred = scaled_logit_model.predict(X_test)
print("accuracy: ", accuracy_score(y_test, y_pred) *100, "%")
print("precision: ", precision_score(y_test, y_pred)*100, "%")

accuracy:  86.29629629629629 %
precision:  86.66666666666667 %


In [29]:
## Regularization (L1 & l2)

#Lasso Regression
from sklearn.linear_model import LogisticRegressionCV

# 4. Instantiate and fit the Lasso Logistic Regression CV model
# Note: 'saga' or 'liblinear' solvers must be used to support L1 penalties
lasso_logit_cv = LogisticRegressionCV(
    penalty='l1', 
    solver='saga', 
    cv=5, 
    random_state=42, 
    max_iter=10000
)
lasso_logit_cv.fit(X_train, y_train)

# 5. Output the best inverse regularization strength (C)
print(f"Optimal C value: {lasso_logit_cv.C_[0]}")

# 6. Check which feature coefficients were shrunk exactly to zero
print(f"Model Coefficients: {lasso_logit_cv.coef_[0]}")

Optimal C value: 0.046415888336127774
Model Coefficients: [ 1.10728912  1.047949    0.82212711  0.99372978  1.14411072  0.
 -0.03148478  0.          0.          0.          0.          0.
  0.          0.          0.        ]


In [30]:
# Evaluating L1 model
y_pred = lasso_logit_cv.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("accuracy: ", accuracy)
precision = precision_score(y_test, y_pred)
print("precision: ", precision)

accuracy:  0.8555555555555555
precision:  0.8583333333333333


In [31]:
# L2 / Ridge Regression
from sklearn.linear_model import LogisticRegressionCV
ridge_logit_cv = LogisticRegressionCV(
    penalty = 'l2',
    solver = 'lbfgs', # used for dataset having rows upto 100k
    cv = 5,
    random_state = 42,
    max_iter = 10000
)
#train data on best C (c=1/alpha) value found from ridge regression
ridge_logit_cv.fit(X_train, y_train)
#predict values using the trained ridge regression cv model
y_predict = ridge_logit_cv.predict(X_test)

# 5. Output the best inverse regularization strength (C)
print(f"Optimal C value: {ridge_logit_cv.C_[0]}")

# 6. Check which feature coefficients were shrunk NOT exactly to zero, near zero
print(f"Model Coefficients: {ridge_logit_cv.coef_[0]}")
    

Optimal C value: 0.046415888336127774
Model Coefficients: [ 1.08437017  1.0387929   0.84094162  1.00002365  1.12133128 -0.09593081
 -0.15252836 -0.00290667 -0.04362072  0.0529399  -0.00971748  0.0447497
 -0.06614957  0.07246207 -0.01310172]


In [32]:
# Evaluate ridge regression model
print("accuracy: ", accuracy_score(y_test, y_pred)*100,"\n precision: ", precision_score(y_test,y_pred))

accuracy:  85.55555555555556 
 precision:  0.8583333333333333


In [34]:
# Model Evaluation
from sklearn.metrics import accuracy_score, classification_report

models = {'Baseline': logistic_regr_model, 'Scaled Baseline': scaled_logit_model, 'Lasso': lasso_logit_cv, 'Ridge': ridge_logit_cv}

for name, model in models.items():
    if(name =='Baseline'):
        y_pred = model.predict(X_test_unscale)
    else:
        y_pred = model.predict(X_test)
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))


Baseline
Accuracy: 0.8592592592592593
              precision    recall  f1-score   support

           0       0.85      0.90      0.87       145
           1       0.87      0.82      0.84       125

    accuracy                           0.86       270
   macro avg       0.86      0.86      0.86       270
weighted avg       0.86      0.86      0.86       270


Scaled Baseline
Accuracy: 0.8629629629629629
              precision    recall  f1-score   support

           0       0.86      0.89      0.87       145
           1       0.87      0.83      0.85       125

    accuracy                           0.86       270
   macro avg       0.86      0.86      0.86       270
weighted avg       0.86      0.86      0.86       270


Lasso
Accuracy: 0.8555555555555555
              precision    recall  f1-score   support

           0       0.85      0.88      0.87       145
           1       0.86      0.82      0.84       125

    accuracy                           0.86       270
   macr

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Get the C values tested by the model
c_values = lasso_logit_cv.Cs_

# 2. Extract accuracy scores and average them across the 5 cross-validation folds
# The shape of .scores_[1.0] is (n_folds, n_Cs) for binary classification
# We average across axis 0 (the folds)
mean_accuracy_path = lasso_logit_cv.scores_[1.0].mean(axis=0)

# 3. Find the optimal C value determined by the model
optimal_c = lasso_logit_cv.C_[0]

# plot
plt.figure(figsize=(10, 6))

# Plot C values on a logarithmic scale
plt.semilogx(c_values, mean_accuracy_path, marker='o', color='g', linewidth=2)

# Highlight the optimal C value with a vertical line
plt.axvline(x=optimal_c, color='r', linestyle='--', label=f'Optimal C ({optimal_c:.4f})')

plt.title('Lasso Logistic Regression: Accuracy vs Inverse Regularization Strength (C)', fontsize=14)
plt.xlabel('Inverse Regularization Strength (C) - Log Scale', fontsize=12)
plt.ylabel('Mean CV Accuracy', fontsize=12)
plt.grid(True, which="both", ls="-", alpha=0.5)
plt.legend()
plt.show()

